# Lab 8: Multi Tenancy

## Hands-on Lab

In this lab, students will examine how multitenancy can be implemented on shared infrastructure.

---

## Objectives

By the end of this lab, students will be able to:

- Describe the role of an MCP gateway in a shared service architecture
- Inspect `LibreChat` MCP configuration and gateway routing
- Compare authenticated and unauthenticated MCP gateway targets
- Review identity claims used for tenant-aware authorization

## MCP Gateway

An MCP gateway is an intermediary layer between an application and one or more MCP servers.

Instead of the application connecting directly to every MCP server, it connects to the gateway. The gateway can then route requests, enforce access control, manage authentication, normalize server behavior, log activity, and provide a single entry point for multiple MCP tools or services.

In practice, it works like a reverse proxy or API gateway for MCP:

```text
Application → MCP Gateway → MCP Servers
```

In this lab, [`agentgateway`](https://agentgateway.dev) is used by [`LibreChat`](https://www.librechat.ai) to connect to the MCP servers deployed in the lab.

```text
LibreChat → agentgateway → MCP servers (k8s, calendar, booking, etc.)
```

`LibreChat` is available at [https://librechat.lab.control-plane.io](https://librechat.lab.control-plane.io).
Inspect its configuration before reviewing the gateway targets.

> **Tip:** Use `mcp-inspector` to observe MCP server behavior from the client side before changing any configuration.

Open `mcp-inspector` with the lab certificate bundle configured:

```bash
NODE_EXTRA_CA_CERTS="/etc/ssl/certs/ca-certificates.crt" mcp-inspector
```

In [ ]:
%%bash

kubectl describe configmap --namespace librechat librechat-librechat-config

Focus on the `mcpServers` section.

```yaml
mcpServers:
    agentgateway-mcps:
        timeout: 300000
        type: streamable-http
        url: "https://agentgateway-proxy.lab.control-plane.io"
        startup: false
        headers:
            X-User-ID: "{{LIBRECHAT_USER_ID}}"
            X-User-Email: "{{LIBRECHAT_USER_EMAIL}}"
            X-User-Username: "{{LIBRECHAT_USER_USERNAME}}"
            X-User-Role: "{{LIBRECHAT_USER_ROLE}}"
            x-user-id-token: "{{LIBRECHAT_OPENID_TOKEN}}"
            x-user-id-idtoken: "{{LIBRECHAT_OPENID_ID_TOKEN}}"
    agentgateway-mcps-protected:
        timeout: 300000
        type: streamable-http
        url: "https://agentgateway-proxy-protected.lab.control-plane.io/mcp"
        startup: false
        oauth:
            authorization_url: "https://idm.lab.control-plane.io/application/o/authorize/"
            token_url: "https://idm.lab.control-plane.io/application/o/token/"
            client_id: "agentgateway"
            client_secret: "${MCP_CLIENT_SECRET}"
            redirect_uri: "https://librechat.lab.control-plane.io/api/mcp/agentgateway-mcps-protected/oauth/callback"
            scope: "openid profile email offline_access"

            grant_types_supported:
                - authorization_code
                - refresh_token
            response_types_supported:
                - code
            token_endpoint_auth_methods_supported:
                - client_secret_basic
            code_challenge_methods_supported:
                - S256
```

The configuration defines two targets:

- `agentgateway-mcp` connects to [the unprotected agentgateway proxy](https://agentgateway-proxy.lab.control-plane.io). No authentication is performed, and selected `LibreChat` user attributes are passed to the backend through headers.
- `agentgateway-mcp-protected` connects to [the protected agentgateway proxy](https://agentgateway-proxy-protected.lab.control-plane.io/mcp). This target initializes an OAuth flow.

> **Question:** Which headers or tokens should a backend trust when multiple tenants share the same gateway?

For `agentgateway-mcp-protected`:

- `authorization_url`: The endpoint where the user is redirected to start login and consent. The application sends the user here with parameters like `client_id`, `redirect_uri`, `scope`, and `response_type`. After successful authentication, the identity provider redirects back with an authorization code.
- `token_url`: The endpoint the application calls server-to-server to exchange the authorization code for tokens. It usually returns an access token, ID token, refresh token, or some subset of those.
- `client_id`: The public identifier of the OAuth client registered with the identity provider. It tells the identity provider which application is requesting authentication.
- `client_secret`: The confidential credential for the OAuth client. The value is read from the environment variable `MCP_CLIENT_SECRET`, rather than hardcoded. It proves that the backend exchanging the code is the registered client.
- `redirect_uri`: The callback URL where the identity provider sends the user after login. This must usually match exactly one of the redirect URIs registered for the OAuth client. The application receives the authorization code here.
- `scope`: The permissions and identity claims requested.
    - `openid` means this is an OpenID Connect login, not just plain OAuth.
    - `profile` requests basic user profile claims, such as name or username.
    - `email` requests the user's email claim.
    - `offline_access` requests a refresh token, allowing the application to obtain new access tokens without requiring the user to log in again.

Inspect the shape of an access token in the lab identity provider, [`authentik`](https://goauthentik.io/).

Navigate to the provider page in `authentik` and sign in with the `admin` credentials.

[Authentik provider configuration](https://idm.lab.control-plane.io/if/admin/#/core/providers)

```text
user: admin
password: Password123
```

![Authentik providers page](./img/provider.png)

Inspect the `agentgateway` provider.

![agentgateway provider configuration](./img/agentgateway-provider.png)

Preview the JWT generated when `admin` signs in to this provider.

![JWT preview for admin user](./img/admin-jwt.png)

Compare the claims for an administrator and a non-administrator user.

![JWT preview for alice user](./img/alice-jwt.png)

> **Question:** Which claim changes would be required for the gateway or backend MCP server to distinguish tenant membership from administrative access?

Preview tokens for other users to compare user attributes and group membership.